In [2]:
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from rca import process_categorical, best_logistic_solver, checker
from tqdm.notebook import tqdm

In [3]:
top_text = 'BERT_base'

# Loading best performing text embedding from RCA
text = pd.read_csv(f'../../data/embeds/{top_text}.csv', index_col=0, header=None)

# Subsetting to brain_behavior_union
with open('../../data/brain_behav_union.pkl', 'rb') as f:
    brain_behav_union = pickle.load(f)
    text = text.loc[text.index.intersection(brain_behav_union)]

# Standardizing
text = (text - text.mean()) / text.std()
text

,1,2,3,4,5,6,7,8,9,10,...,759,760,761,762,763,764,765,766,767,768
0,,,,,,,,,,,,,,,,,,,,,
prose,-0.433234,0.991354,-0.199200,0.100646,-1.531259,0.316878,1.391669,-0.623537,0.735869,-0.502189,...,1.429784,-0.883397,-1.569495,0.518099,-0.443259,0.500011,0.762343,0.817947,1.045552,-1.188028
rend,-0.293489,-0.040316,-0.697051,0.660044,-0.240452,0.179847,-0.101866,-0.832965,0.501559,-1.218700,...,0.860490,-0.407320,-0.269539,-0.984510,-1.071554,1.399868,-0.128909,0.217632,-0.279376,-0.512413
priest,1.087055,1.273142,0.496819,-2.349012,-2.214224,-0.132140,1.119795,0.812897,1.850977,-0.819676,...,0.641972,-0.889603,-0.305063,-1.076220,-0.044928,-0.761356,-0.390978,2.122327,2.746881,-0.527276
degradation,-0.078858,0.843897,-1.434848,0.609538,0.550691,-0.655007,0.921856,-0.074975,-0.806146,-0.285324,...,1.470825,-0.282069,0.068405,0.836456,0.958771,-0.469868,0.618132,0.000942,-0.586460,-0.908142
badge,1.085607,0.258633,0.436371,-1.252050,-0.006714,-1.475116,1.721674,-2.407598,0.597143,-0.159273,...,0.342638,0.684054,-1.056262,0.477259,1.429564,0.414533,-1.778269,-0.054080,0.158404,1.028200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
munchkin,-0.896581,-1.832834,1.062427,-0.289412,0.504635,0.715974,-0.365569,0.309989,-1.615336,2.165702,...,-3.341707,1.991853,-0.881123,-0.531908,-0.514896,0.335332,-0.771712,1.245054,1.177842,1.686058
gaunt,0.009937,-0.851149,0.705734,-1.534014,0.454182,1.098658,-0.653353,-1.590118,-1.033217,-1.249994,...,-0.648723,0.488913,1.153627,-2.132417,0.907863,0.015479,-0.893977,1.487130,0.463538,-1.193342
lupus,1.497121,-3.265141,0.458512,-1.026870,1.078593,-0.434067,-0.311114,1.783715,-2.960871,0.310695,...,-0.988946,-0.225404,1.272432,1.016156,-0.169474,2.939869,-0.321230,1.646880,-1.755154,1.034721


In [4]:
# Loading norm data
norms = pd.read_csv('../../data/psychNorms/psychNorms_processed.zip', index_col=0, compression='zip', low_memory=False)
norm_meta = pd.read_csv('../../data/psychNorms/psychNorms_metadata_processed.csv', index_col='norm')
norms

,frequency_lund,frequency_kucera,frequency_subtlexus,frequency_subtlexuk,frequency_blog_gimenes,frequency_twitter_gimenes,frequency_news_gimenes,frequency_written_cobuild,frequency_spoken_cobuild,context_diversity_subtlexus,...,person_vanarsdall,goals_vanarsdall,movement_vanarsdall,concreteness_vanarsdall,familiarity_vanarsdall,imageability_vanarsdall,familiarity_fear,aoa_fear,imageability_fear,sensory_experience_juhasz2013
word,,,,,,,,,,,,,,,,,,,,,
'em,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.3617,1.9138,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'neath,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
're,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.9031,1.6335,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'shun,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'tis,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.4771,0.6021,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
shrick,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.62,4.38,2.93,NaN
post office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.79,3.07,5.29,NaN
fishing rod,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.29,3.38,5.64,NaN


In [6]:
# Ridge
min_ord, max_ord = -5, 5
alphas = np.logspace(min_ord, max_ord, max_ord - min_ord + 1)
ridge = RidgeCV(alphas=alphas)

# Logistic hyperparameters
Cs = 1 / alphas
inner_cv = 5
penalty = 'l2'

# Outer cv setting
outer_cv = 5
n_jobs = 5

# Iterate over each norm, compute the residuals
text_resids = pd.DataFrame(index=norms.index, columns=norms.columns)
for norm_name in tqdm(norms.columns):

    # Aligning embed with norm
    y_true = norms[norm_name].dropna()
    X, y_true = text.align(y_true, axis='index', join='inner', copy=True)

    # Getting norm dtype
    norm_dtype = norm_meta.loc[norm_name, 'type']

    # Estimator: ridge or logistic
    if norm_dtype in ['binary', 'multiclass']:
        X, y_true = process_categorical(outer_cv, inner_cv, X, y_true)

        # may have switched form multi to bin after processing
        norm_dtype = 'binary' if len(y_true.unique()) == 2 else 'multiclass'

        # logistic regression settings
        solver = best_logistic_solver(y_true, norm_dtype)
        estimator = LogisticRegressionCV(
            Cs=Cs,
            penalty=penalty,
            cv=StratifiedKFold(inner_cv),
            solver=solver
        )
        predict_method = 'predict_proba'
    else:
        estimator = ridge
        predict_method = 'predict'

        # Run cross_val_predict (UNINDENT FOR CATEGORICAL NORMS)
        associated_embed = norm_meta.loc[norm_name, 'associated_embed']
        check = checker(top_text, y_true, norm_dtype, associated_embed, outer_cv)
        if check == 'pass' and len(y_true) > outer_cv:
            y_pred = cross_val_predict(estimator, X, y_true, cv=outer_cv, n_jobs=n_jobs, method=predict_method)
            text_resids[norm_name] = pd.Series(y_true - y_pred, index=y_true.index)

text_resids.to_csv('../../data/results/text_resids.csv')
text_resids

  0%|          | 0/291 [00:00<?, ?it/s]

,frequency_lund,frequency_kucera,frequency_subtlexus,frequency_subtlexuk,frequency_blog_gimenes,frequency_twitter_gimenes,frequency_news_gimenes,frequency_written_cobuild,frequency_spoken_cobuild,context_diversity_subtlexus,...,person_vanarsdall,goals_vanarsdall,movement_vanarsdall,concreteness_vanarsdall,familiarity_vanarsdall,imageability_vanarsdall,familiarity_fear,aoa_fear,imageability_fear,sensory_experience_juhasz2013
word,,,,,,,,,,,,,,,,,,,,,
'em,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'neath,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
're,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'shun,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'tis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
shrick,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
post office,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fishing rod,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
